In [1]:
import argparse
import os
import pathlib
import sys
from functools import reduce

import duckdb
import pandas as pd
from image_analysis_3D.file_utils.arg_parsing_utils import parse_args
from image_analysis_3D.file_utils.notebook_init_utils import (
    bandicoot_check,
    init_notebook,
)

root_dir, in_notebook = init_notebook()

profile_base_dir = bandicoot_check(
    pathlib.Path(os.path.expanduser("~/mnt/bandicoot/NF1_organoid_data")).resolve(),
    root_dir,
)
profile_base_dir = root_dir

In [2]:
if not in_notebook:
    args = parse_args()
    well_fov = args["well_fov"]
    patient = args["patient"]
    output_features_subparent_name = args["output_features_subparent_name"]
    image_based_profiles_subparent_name = args["image_based_profiles_subparent_name"]


else:
    well_fov = "E9-1"
    patient = "NF0014_T1"
    output_features_subparent_name = "extracted_features"
    image_based_profiles_subparent_name = "image_based_profiles"


result_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{output_features_subparent_name}/{well_fov}"
).resolve(strict=True)
database_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/0.converted_profiles/{well_fov}"
).resolve()
database_path.mkdir(parents=True, exist_ok=True)
# create the sqlite database
sqlite_path = database_path / f"{well_fov}.duckdb"
DB_structure_path = pathlib.Path(
    f"{root_dir}/4.processing_image_based_profiles/data/DB_structures/DB_structure_db.duckdb"
).resolve(strict=True)

# get a list of all parquets in the directory recursively
parquet_files = list(result_path.rglob("*.parquet"))
parquet_files.sort()
print(len(parquet_files), "parquet files found")
if len(parquet_files) != 101:
    raise ValueError(f"Expected 101 parquet files, but found {len(parquet_files)}")

101 parquet files found


In [3]:
# create the nested dictionary to hold the feature types and compartments
feature_types = [
    "AreaSizeShape",
    "Colocalization",
    "Intensity",
    "Granularity",
    "Neighbors",
    "SAMMed3D",
    "Texture",
    "CHAMMI75",
]
compartments = ["Organoid", "Nuclei", "Cell", "Cytoplasm", "Nucleocentric"]

In [4]:
output_dict = {
    compartment: {
        feature_type: []
        for feature_type in feature_types
        if not (
            compartment == "Nucleocentric"
            and feature_type.lower() not in ["chammi75", "sammed3d"]
        )
    }
    for compartment in compartments
}
output_dict

{'Organoid': {'AreaSizeShape': [],
  'Colocalization': [],
  'Intensity': [],
  'Granularity': [],
  'Neighbors': [],
  'SAMMed3D': [],
  'Texture': [],
  'CHAMMI75': []},
 'Nuclei': {'AreaSizeShape': [],
  'Colocalization': [],
  'Intensity': [],
  'Granularity': [],
  'Neighbors': [],
  'SAMMed3D': [],
  'Texture': [],
  'CHAMMI75': []},
 'Cell': {'AreaSizeShape': [],
  'Colocalization': [],
  'Intensity': [],
  'Granularity': [],
  'Neighbors': [],
  'SAMMed3D': [],
  'Texture': [],
  'CHAMMI75': []},
 'Cytoplasm': {'AreaSizeShape': [],
  'Colocalization': [],
  'Intensity': [],
  'Granularity': [],
  'Neighbors': [],
  'SAMMed3D': [],
  'Texture': [],
  'CHAMMI75': []},
 'Nucleocentric': {'SAMMed3D': [], 'CHAMMI75': []}}

In [5]:
files = list(result_path.rglob("*.parquet"))
files_df = pd.DataFrame({"file_path": files})
files_df["file_name"] = files_df["file_path"].apply(lambda x: x.name)
files_df["compartment"] = files_df["file_name"].apply(lambda x: x.split("_")[0])
files_df["channel"] = files_df["file_name"].apply(lambda x: x.split("_")[1])
files_df["feature_type"] = files_df["file_name"].apply(
    lambda x: x.split("_")[2].split(".parquet")[0]
)
file_path = files_df.pop("file_path")
files_df.insert(4, "file_path", file_path)
files_df.head()

,file_name,compartment,channel,feature_type,file_path
0,Nuclei_ER_SAMMed3D_GPU_features.parquet,Nuclei,ER,SAMMed3D,/home/lippincm/Documents/NF1_3D_organoid_profi...
1,Cell_AGP_Granularity_CPU_features.parquet,Cell,AGP,Granularity,/home/lippincm/Documents/NF1_3D_organoid_profi...
2,Cytoplasm_Mito_Intensity_CPU_features.parquet,Cytoplasm,Mito,Intensity,/home/lippincm/Documents/NF1_3D_organoid_profi...
3,Nuclei_AGP_Intensity_CPU_features.parquet,Nuclei,AGP,Intensity,/home/lippincm/Documents/NF1_3D_organoid_profi...
4,Organoid_AGP_Granularity_CPU_features.parquet,Organoid,AGP,Granularity,/home/lippincm/Documents/NF1_3D_organoid_profi...


In [6]:
for i, row in files_df.iterrows():
    compartment = row["compartment"]
    feature_type = row["feature_type"]
    channel = row["channel"]
    file_path = row["file_path"]
    output_dict[compartment][feature_type].append(file_path)
output_dict

{'Organoid': {'AreaSizeShape': [PosixPath('/home/lippincm/Documents/NF1_3D_organoid_profiling_pipeline/data/NF0014_T1/extracted_features/E9-1/Organoid_NoChannel_AreaSizeShape_CPU_features.parquet')],
  'Colocalization': [PosixPath('/home/lippincm/Documents/NF1_3D_organoid_profiling_pipeline/data/NF0014_T1/extracted_features/E9-1/Organoid_DNA-ER_Colocalization_CPU_features.parquet'),
   PosixPath('/home/lippincm/Documents/NF1_3D_organoid_profiling_pipeline/data/NF0014_T1/extracted_features/E9-1/Organoid_DNA-Mito_Colocalization_CPU_features.parquet'),
   PosixPath('/home/lippincm/Documents/NF1_3D_organoid_profiling_pipeline/data/NF0014_T1/extracted_features/E9-1/Organoid_ER-Mito_Colocalization_CPU_features.parquet'),
   PosixPath('/home/lippincm/Documents/NF1_3D_organoid_profiling_pipeline/data/NF0014_T1/extracted_features/E9-1/Organoid_Mito-AGP_Colocalization_CPU_features.parquet'),
   PosixPath('/home/lippincm/Documents/NF1_3D_organoid_profiling_pipeline/data/NF0014_T1/extracted_featur

In [47]:
output_dict["Nucleocentric"]

{'SAMMed3D': [PosixPath('/home/lippincm/Documents/NF1_3D_organoid_profiling_pipeline/data/NF0014_T1/extracted_features/E9-1/Nucleocentric_Mito_SAMMed3D_GPU_features.parquet'),
  PosixPath('/home/lippincm/Documents/NF1_3D_organoid_profiling_pipeline/data/NF0014_T1/extracted_features/E9-1/Nucleocentric_ER_SAMMed3D_GPU_features.parquet'),
  PosixPath('/home/lippincm/Documents/NF1_3D_organoid_profiling_pipeline/data/NF0014_T1/extracted_features/E9-1/Nucleocentric_AGP_SAMMed3D_GPU_features.parquet'),
  PosixPath('/home/lippincm/Documents/NF1_3D_organoid_profiling_pipeline/data/NF0014_T1/extracted_features/E9-1/Nucleocentric_DNA_SAMMed3D_GPU_features.parquet')],
 'CHAMMI75': [PosixPath('/home/lippincm/Documents/NF1_3D_organoid_profiling_pipeline/data/NF0014_T1/extracted_features/E9-1/Nucleocentric_DNA_CHAMMI75_GPU_features.parquet'),
  PosixPath('/home/lippincm/Documents/NF1_3D_organoid_profiling_pipeline/data/NF0014_T1/extracted_features/E9-1/Nucleocentric_Mito_CHAMMI75_GPU_features.parquet

In [ ]:
final_df_dict = {compartment: {} for compartment in output_dict.keys()}
for compartment in output_dict.keys():
    for feature_type in output_dict[compartment].keys():
        if not output_dict[compartment][feature_type]:
            continue
        final_df_dict[compartment][feature_type] = reduce(
            lambda left, right: pd.merge(
                left,
                right,
                on=["object_id", "image_set"],
                how="left",
            ),
            [pd.read_parquet(file) for file in output_dict[compartment][feature_type]],
        )
# ensure the object_id column is int before merging
for compartment in final_df_dict.keys():
    for feature_type in final_df_dict[compartment].keys():
        final_df_dict[compartment][feature_type]["object_id"] = final_df_dict[
            compartment
        ][feature_type]["object_id"].astype(int)

In [51]:
final_df_dict["Nucleocentric"]["SAMMed3D"]

,object_id,image_set,Nucleocentric_Mito_CHAMMI75_Feature0,Nucleocentric_Mito_CHAMMI75_Feature1,Nucleocentric_Mito_CHAMMI75_Feature10,Nucleocentric_Mito_CHAMMI75_Feature100,Nucleocentric_Mito_CHAMMI75_Feature101,Nucleocentric_Mito_CHAMMI75_Feature102,Nucleocentric_Mito_CHAMMI75_Feature103,Nucleocentric_Mito_CHAMMI75_Feature104,...,Nucleocentric_DNA_SAMMed3D_Feature90,Nucleocentric_DNA_SAMMed3D_Feature91,Nucleocentric_DNA_SAMMed3D_Feature92,Nucleocentric_DNA_SAMMed3D_Feature93,Nucleocentric_DNA_SAMMed3D_Feature94,Nucleocentric_DNA_SAMMed3D_Feature95,Nucleocentric_DNA_SAMMed3D_Feature96,Nucleocentric_DNA_SAMMed3D_Feature97,Nucleocentric_DNA_SAMMed3D_Feature98,Nucleocentric_DNA_SAMMed3D_Feature99
0,257,E9-1,4.163089,-0.909741,0.539815,2.559543,0.119701,2.536916,2.289842,-2.520163,...,-0.007136,-0.050355,0.074250,-0.011046,0.016181,-0.065276,-0.113449,0.251154,0.341022,0.186649
1,514,E9-1,3.737707,-1.072771,3.544111,1.045316,-0.253289,2.919431,4.003189,-1.292669,...,-0.007064,-0.177103,0.064973,-0.010797,0.042807,-0.100086,0.030172,0.158761,0.354477,0.086128
2,771,E9-1,1.223940,0.595523,-0.261445,1.550906,-1.552259,4.213758,1.900936,-2.600322,...,-0.007628,-0.044639,0.042606,-0.010729,0.009675,-0.042756,-0.063998,0.216499,0.399424,0.182785
3,1285,E9-1,1.132286,-1.475073,1.012799,1.425551,-0.406091,0.720181,-0.450617,-1.883056,...,-0.007078,-0.066354,0.068478,-0.010734,0.010313,-0.063647,-0.056182,0.254585,0.337616,0.188783
4,1542,E9-1,3.574742,-0.808908,2.019418,0.285833,-2.775232,2.351737,2.334319,-0.803855,...,-0.007773,-0.051926,0.167763,-0.010880,0.012455,-0.125910,-0.045071,0.121093,0.403455,0.167635
5,1799,E9-1,1.442207,-2.732410,1.177282,0.571325,1.399111,1.592824,2.864717,0.262707,...,-0.007577,-0.058811,0.078695,-0.010628,0.016391,-0.027385,-0.046834,0.233832,0.339102,0.167775
6,2056,E9-1,2.643584,-2.437306,3.120502,-0.990144,-2.376880,1.122741,2.832258,0.278236,...,-0.007504,-0.087710,0.091699,-0.010816,0.045311,-0.095006,-0.080825,0.232428,0.345474,0.189369
7,2313,E9-1,-0.309827,1.975163,-1.053658,2.271537,0.098605,2.776303,1.638288,-0.553274,...,-0.005967,-0.074876,0.071930,-0.010702,0.022353,0.015701,0.001315,0.254712,0.306036,0.229803
8,2570,E9-1,1.010511,-0.205042,1.062148,0.461221,0.061294,5.251267,3.180134,-0.968768,...,-0.006304,-0.104988,0.094308,-0.011003,0.019935,-0.030390,-0.033140,0.239339,0.335525,0.179282
9,2827,E9-1,4.745710,-0.305881,1.841763,2.018081,0.791441,3.166679,3.432388,-2.207949,...,-0.008176,-0.082077,0.051097,-0.010106,0.037340,-0.060246,0.099444,0.117454,0.284037,0.088279


In [43]:
# merge the dfs such that each compartment has a single df with all feature types as columns
compartment_dfs = {}
for compartment in final_df_dict.keys():
    for df in final_df_dict[compartment].values():
        # if compartment == "Nucleocentric":
        #     compartment_dfs[compartment] = df
        #     break

        compartment_dfs[compartment] = reduce(
            lambda left, right: pd.merge(
                left,
                right,
                on=["object_id", "image_set"],
                how="left",
            ),
            final_df_dict[compartment].values(),
        )

In [44]:
# assert that the number of Nuclei, Cell, Cytoplasm, Nucleocentric are all the same
print(
    len(compartment_dfs["Nuclei"]),
    len(compartment_dfs["Cell"]),
    len(compartment_dfs["Cytoplasm"]),
    len(compartment_dfs["Nucleocentric"]),
)
assert (
    len(compartment_dfs["Nuclei"])
    == len(compartment_dfs["Cell"])
    == len(compartment_dfs["Cytoplasm"])
    == len(compartment_dfs["Nucleocentric"])
)

# assert that all object ids line up across compartments
assert (
    compartment_dfs["Nuclei"]["object_id"].equals(compartment_dfs["Cell"]["object_id"])
    and compartment_dfs["Nuclei"]["object_id"].equals(
        compartment_dfs["Cytoplasm"]["object_id"]
    )
    and compartment_dfs["Nuclei"]["object_id"].equals(
        compartment_dfs["Nucleocentric"]["object_id"]
    )
)

32 32 32 32


In [45]:
compartment_dfs["Nucleocentric"]

,object_id,image_set,Nucleocentric_Mito_CHAMMI75_Feature0_x,Nucleocentric_Mito_CHAMMI75_Feature1_x,Nucleocentric_Mito_CHAMMI75_Feature10_x,Nucleocentric_Mito_CHAMMI75_Feature100_x,Nucleocentric_Mito_CHAMMI75_Feature101_x,Nucleocentric_Mito_CHAMMI75_Feature102_x,Nucleocentric_Mito_CHAMMI75_Feature103_x,Nucleocentric_Mito_CHAMMI75_Feature104_x,...,Nucleocentric_ER_CHAMMI75_Feature90_y,Nucleocentric_ER_CHAMMI75_Feature91_y,Nucleocentric_ER_CHAMMI75_Feature92_y,Nucleocentric_ER_CHAMMI75_Feature93_y,Nucleocentric_ER_CHAMMI75_Feature94_y,Nucleocentric_ER_CHAMMI75_Feature95_y,Nucleocentric_ER_CHAMMI75_Feature96_y,Nucleocentric_ER_CHAMMI75_Feature97_y,Nucleocentric_ER_CHAMMI75_Feature98_y,Nucleocentric_ER_CHAMMI75_Feature99_y
0,257,E9-1,4.163089,-0.909741,0.539815,2.559543,0.119701,2.536916,2.289842,-2.520163,...,0.772643,5.292253,-11.560932,2.214991,5.065310,-2.337864,0.853876,3.678142,1.301484,-1.591434
1,514,E9-1,3.737707,-1.072771,3.544111,1.045316,-0.253289,2.919431,4.003189,-1.292669,...,1.230746,6.550419,-10.415889,4.427794,2.895350,-0.918173,-2.929753,3.686656,1.196309,-1.429748
2,771,E9-1,1.223940,0.595523,-0.261445,1.550906,-1.552259,4.213758,1.900936,-2.600322,...,-1.953734,6.983423,-6.470373,6.091298,-0.854538,1.003946,-1.052528,4.367959,-3.134054,-2.874774
3,1285,E9-1,1.132286,-1.475073,1.012799,1.425551,-0.406091,0.720181,-0.450617,-1.883056,...,0.741673,3.478837,-5.227507,6.752029,0.123695,3.283154,-2.657704,2.571409,-3.321669,-2.781133
4,1542,E9-1,3.574742,-0.808908,2.019418,0.285833,-2.775232,2.351737,2.334319,-0.803855,...,1.707944,4.549797,-3.489297,6.379190,-3.014529,4.498521,-4.837077,4.436558,1.575774,-0.253532
5,1799,E9-1,1.442207,-2.732410,1.177282,0.571325,1.399111,1.592824,2.864717,0.262707,...,-0.403066,4.035362,-5.553319,8.353971,-0.053020,0.462281,-0.853180,5.318773,-4.143157,-1.343526
6,2056,E9-1,2.643584,-2.437306,3.120502,-0.990144,-2.376880,1.122741,2.832258,0.278236,...,0.108406,5.560587,-6.107862,7.789940,3.009899,3.438133,-0.975063,3.404965,-3.471829,-0.503241
7,2313,E9-1,-0.309827,1.975163,-1.053658,2.271537,0.098605,2.776303,1.638288,-0.553274,...,-1.832377,5.927807,-12.283434,-0.646739,2.273872,-2.244855,1.008664,5.524864,-0.751357,-2.016670
8,2570,E9-1,1.010511,-0.205042,1.062148,0.461221,0.061294,5.251267,3.180134,-0.968768,...,0.704557,4.299072,-10.926056,-0.220481,3.679306,-3.864789,0.787039,2.879037,-2.291432,-1.569307
9,2827,E9-1,4.745710,-0.305881,1.841763,2.018081,0.791441,3.166679,3.432388,-2.207949,...,-0.030813,2.789876,-9.114958,4.489854,1.673064,0.954411,-3.508823,4.099704,1.295795,-2.921562


In [39]:
nuclei_ids = compartment_dfs["Nuclei"]["object_id"].unique()
nuclei_ids = [x * 257 for x in nuclei_ids]
cell_ids = compartment_dfs["Cell"]["object_id"].unique()

print(nuclei_ids)
print(cell_ids)
set(nuclei_ids) - set(cell_ids)

[np.int64(66049), np.int64(132098), np.int64(198147), np.int64(330245), np.int64(396294), np.int64(462343), np.int64(528392), np.int64(594441), np.int64(660490), np.int64(726539), np.int64(792588), np.int64(858637), np.int64(924686), np.int64(990735), np.int64(1056784), np.int64(1122833), np.int64(1188882), np.int64(1254931), np.int64(1387029), np.int64(1453078), np.int64(1585176), np.int64(1651225), np.int64(1717274), np.int64(1783323), np.int64(1849372), np.int64(1915421), np.int64(1981470), np.int64(2047519), np.int64(2113568), np.int64(2179617), np.int64(2245666), np.int64(2311715)]
[ 257  514  771 1285 1542 1799 2056 2313 2570 2827 3084 3341 3598 3855
 4112 4369 4626 4883 5397 5654 6168 6425 6682 6939 7196 7453 7710 7967
 8224 8481 8738 8995]


{np.int64(66049),
 np.int64(132098),
 np.int64(198147),
 np.int64(330245),
 np.int64(396294),
 np.int64(462343),
 np.int64(528392),
 np.int64(594441),
 np.int64(660490),
 np.int64(726539),
 np.int64(792588),
 np.int64(858637),
 np.int64(924686),
 np.int64(990735),
 np.int64(1056784),
 np.int64(1122833),
 np.int64(1188882),
 np.int64(1254931),
 np.int64(1387029),
 np.int64(1453078),
 np.int64(1585176),
 np.int64(1651225),
 np.int64(1717274),
 np.int64(1783323),
 np.int64(1849372),
 np.int64(1915421),
 np.int64(1981470),
 np.int64(2047519),
 np.int64(2113568),
 np.int64(2179617),
 np.int64(2245666),
 np.int64(2311715)}

In [40]:
with duckdb.connect(DB_structure_path, read_only=True) as cx:
    organoid_table = cx.execute("SELECT * FROM Organoid").df()
    cell_table = cx.execute("SELECT * FROM Cell").df()
    nuclei_table = cx.execute("SELECT * FROM Nuclei").df()
    cytoplasm_table = cx.execute("SELECT * FROM Cytoplasm").df()
    nucleocentric_table = cx.execute("SELECT * FROM Nucleocentric").df()

dict_of_DB_structues = {
    "Organoid": organoid_table,
    "Cell": cell_table,
    "Nuclei": nuclei_table,
    "Cytoplasm": cytoplasm_table,
    "Nucleocentric": nucleocentric_table,
}

In [41]:
# get the table from the DB_structue
with duckdb.connect(sqlite_path, read_only=False) as cx:
    for compartment, df in compartment_dfs.items():
        print(compartment, df.shape)
        if df.empty:
            cx.register("temp_df", dict_of_DB_structues[compartment])
            cx.execute(
                f"CREATE OR REPLACE TABLE {compartment} AS SELECT * FROM temp_df"
            )
            cx.unregister("temp_df")
        else:
            cx.register("temp_df", df)
            cx.execute(
                f"CREATE OR REPLACE TABLE {compartment} AS SELECT * FROM temp_df"
            )
            cx.unregister("temp_df")

Organoid (1, 3961)
Nuclei (32, 3963)
Cell (32, 3961)
Cytoplasm (32, 3961)
Nucleocentric (32, 4610)
